# Notebook 11 — SPINK7-KLK5 Computational Mutagenesis Scanning

This notebook deploys the alchemical FEP pipeline for systematic
computational mutagenesis of the SPINK7-KLK5 interface.

## Objectives

1. **Binding hotspot identification** — Alanine scanning of interface residues.
2. **EoE variant characterization** — $\Delta\Delta G$ for disease-associated *SPINK7* mutations.
3. **SPINK7-mimetic design** — Identify stabilizing mutations for therapeutic peptide candidates.

## Thermodynamic Cycle

$$\Delta\Delta G = \Delta G_{\text{alch,complex}} - \Delta G_{\text{alch,free}}$$

Each mutation runs two FEP legs (complex + free solution) with 20 $\lambda$ windows × 2 ns each.

In [ ]:
# Cell 1 — Environment Setup
!pip install openmm openmmtools pymbar mdtraj numpy matplotlib

import openmm
import openmmtools
import pymbar
print(f"OpenMM: {openmm.__version__}, openmmtools: {openmmtools.__version__}, pymbar: {pymbar.__version__}")

In [ ]:
# Cell 2 — Project Setup
import sys
from pathlib import Path

USE_COLAB = False
if USE_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/SPINK7_KLK5')
else:
    PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import FEPConfig, KCAL_TO_KJ
from src.simulate.fep import run_fep_campaign
from src.analyze.fep import compute_delta_g_mbar, compute_delta_delta_g

FEP_OUTPUT = PROJECT_ROOT / 'data' / 'analysis' / 'fep' / 'spink7_klk5'
FEP_OUTPUT.mkdir(parents=True, exist_ok=True)

In [ ]:
# Cell 3 — Define Priority Mutations
import numpy as np

MUTATIONS = [
    {
        'name': 'P1_Arg_to_Ala',
        'chain': 'A',  # SPINK7 chain
        'residue_number': None,  # Set after loading structure
        'wild_type': 'ARG',
        'mutant': 'ALA',
        'rationale': 'Test S1 pocket specificity at RSL P1 position',
        'expected': 'Large destabilization (DDG > 5 kcal/mol)',
    },
    {
        'name': 'P2_hydrophobic_to_Ala',
        'chain': 'A',
        'residue_number': None,
        'wild_type': None,
        'mutant': 'ALA',
        'rationale': 'Test P2 sub-site contribution to binding',
        'expected': 'Moderate destabilization',
    },
    {
        'name': 'Cys_to_Ser_disulfide',
        'chain': 'A',
        'residue_number': None,
        'wild_type': 'CYS',
        'mutant': 'SER',
        'rationale': 'Test structural stabilization of RSL by disulfide bonds',
        'expected': 'Destabilization (loss of RSL rigidity)',
    },
]

print(f"Defined {len(MUTATIONS)} priority mutations")
for m in MUTATIONS:
    print(f"  {m['name']}: {m['rationale']}")

In [ ]:
# Cell 4 — Load SPINK7-KLK5 Complex
from openmm.app import PDBFile, ForceField, Modeller
from openmm import unit

complex_pdb_path = PROJECT_ROOT / 'data' / 'pdb' / 'raw' / 'SPINK7_KLK5_docked.pdb'
pdb = PDBFile(str(complex_pdb_path))
forcefield = ForceField('amber14-all.xml', 'amber14/tip3p.xml')

modeller = Modeller(pdb.topology, pdb.positions)
modeller.addHydrogens(forcefield, pH=7.4)
modeller.addSolvent(
    forcefield,
    model='tip3p',
    padding=1.2 * unit.nanometers,
    ionicStrength=0.15 * unit.molar,
)

system = forcefield.createSystem(
    modeller.topology,
    nonbondedMethod=openmm.app.PME,
    nonbondedCutoff=1.0 * unit.nanometers,
    constraints=openmm.app.HBonds,
)

print(f"SPINK7-KLK5 system: {system.getNumParticles()} particles")

In [ ]:
# Cell 5 — Run FEP for Each Mutation (GPU offload)
# Each mutation requires ~4-6 GPU hours on A100

config = FEPConfig(
    n_lambda_windows=20,
    per_window_duration_ns=2.0,
    temperature_k=310.0,
)

results_all = {}

for mutation in MUTATIONS:
    if mutation['residue_number'] is None:
        print(f"Skipping {mutation['name']}: residue_number not set")
        continue

    print(f"\nRunning FEP for {mutation['name']}...")
    mut_output = FEP_OUTPUT / mutation['name']

    # Identify mutant atoms
    mut_indices = []
    for atom in modeller.topology.atoms():
        if (atom.residue.chain.id == mutation['chain']
                and atom.residue.id == str(mutation['residue_number'])):
            mut_indices.append(atom.index)

    if not mut_indices:
        print(f"  WARNING: No atoms found for residue {mutation['residue_number']}")
        continue

    # Complex leg
    complex_dir = mut_output / 'complex'
    complex_result = run_fep_campaign(
        system=system,
        positions=modeller.positions,
        mutant_atom_indices=mut_indices,
        config=config,
        output_dir=complex_dir,
    )

    # Analyze
    dg_complex = compute_delta_g_mbar(
        complex_result['energy_matrix'],
        complex_result['n_samples_per_state'],
        config.temperature_k,
    )

    results_all[mutation['name']] = {
        'dg_complex': dg_complex,
        'mutation': mutation,
    }
    print(f"  DG_complex = {dg_complex['delta_g_kj_mol']:.2f} kJ/mol")

In [ ]:
# Cell 6 — Summary Table
import matplotlib.pyplot as plt

print('=' * 70)
print(f"{'Mutation':<25} {'DDG (kcal/mol)':<18} {'Uncertainty':<15} {'Effect'}")
print('=' * 70)

for name, res in results_all.items():
    if 'ddg' in res:
        ddg = res['ddg']
        effect = 'Destabilizing' if ddg['delta_delta_g_kcal_mol'] > 0 else 'Stabilizing'
        print(f"{name:<25} {ddg['delta_delta_g_kcal_mol']:>+8.2f}        {ddg['delta_delta_g_std_kcal_mol']:>8.2f}         {effect}")

print('=' * 70)
print(f"\nTotal mutations analyzed: {len(results_all)}")

## Interpretation Guidelines

| $\Delta\Delta G$ Range | Interpretation |
|---|---|
| $> +2$ kcal/mol | **Hotspot** — significant destabilization, critical for binding |
| $+0.5$ to $+2$ kcal/mol | Moderate contribution to binding affinity |
| $-0.5$ to $+0.5$ kcal/mol | Negligible effect on binding |
| $< -0.5$ kcal/mol | Stabilizing mutation — candidate for inhibitor design |

## Hardware Requirements

- **Per mutation:** ~4–6 GPU hours on NVIDIA A100 (2 legs × 20 windows × 2 ns)
- **Full scan (3 mutations):** ~12–18 GPU hours

## References

[1] Schreiber & Fersht, *J. Mol. Biol.*, 248, 478–486 (1995)  
[2] Shirts & Chodera, *J. Chem. Phys.*, 129, 124105 (2008)